<a href="https://colab.research.google.com/github/mufidrosidi-404/Komputasi-Numerik/blob/main/%20%20%20materi-A-spl/spl_gauss_jordan_penjelasan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPL 3 Variabel dengan Eliminasi Gauss-Jordan (pivot parsial)

**Mata kuliah:** Komputasi Numerik  
**Nama:** Mufid Rosidi  
**NIM:** 202511420044  
**Materi A:** Sistem Persamaan Linear (SPL) 3 variabel

Notebook ini menyelesaikan SPL tiga variabel dengan cara yang kutulis sendiri: matriks gabungan
kusebut *papan*, lalu diolah dengan Gauss-Jordan. Tiap bagian bisa dijalankan satu per satu
(klik tombol ▶ di sebelah kiri cell), **urutannya dari Cell 1 sampai Cell 9**.

> **Versi penjelasan detail:** kodenya sama persis dengan versi ringkas, tetapi setiap cell diberi penjelasan dan setiap baris kode diberi komentar.

### Cell 1 · Siapkan perkakas

**Penjelasan:** Cell ini cuma memanggil dua pustaka. `random` dipakai di cell uji acak (Cell 9), sedangkan `numpy` hanya kupakai sebagai **pembanding** hasil, bukan untuk menghitung inti. Perhitungan intinya kutulis manual supaya kelihatan langkahnya.

In [ ]:
import random                      # untuk membuat angka acak di Cell 9
import numpy as np                 # hanya untuk membandingkan hasil, bukan untuk hitungan inti
print("Perkakas siap.")            # penanda bahwa cell berhasil jalan


Perkakas siap.


### Cell 2 · Masukkan SPL yang mau diselesaikan

**Penjelasan:** Di sini aku mengisi data soalnya. `koefisien_a` berisi koefisien tiap persamaan, `ruas_kanan_b` berisi angka di kanan tanda sama dengan, dan `nama_peubah` adalah nama tiga variabelnya (aku pakai p, q, r supaya tidak sama dengan x, y, z di buku). Kalau mau ganti soal, cukup ubah tiga variabel ini lalu jalankan ulang cell berikutnya.

In [ ]:
koefisien_a = [                    # tiga persamaan, tiap baris = koefisien p, q, r
    [ 3,  5, -2],                  # 3p + 5q - 2r = -17
    [ 6, -1,  4],                  # 6p - q + 4r = 55
    [-2,  7,  3],                  # -2p + 7q + 3r = -8
]
ruas_kanan_b = [-17, 55, -8]       # angka di sebelah kanan tanda sama dengan
nama_peubah = ["p", "q", "r"]      # nama tiga variabel yang dicari

print("SPL yang akan diselesaikan:")                         # judul tampilan
for baris, kanan in zip(koefisien_a, ruas_kanan_b):         # ambil satu persamaan tiap putaran
    suku = [f"{k:+d}{nama}" for k, nama in zip(baris, nama_peubah)]   # contoh hasil: +3p
    print("  ", " ".join(suku), "=", kanan)                 # cetak persamaan lengkap


SPL yang akan diselesaikan:
   +3p +5q -2r = -17
   +6p -1q +4r = 55
   -2p +7q +3r = -8


### Cell 3 · Fungsi pencetak papan

**Penjelasan:** Fungsi kecil untuk menampilkan *papan* (matriks gabungan koefisien dan ruas kanan) dengan rapi. Garis tegak `|` memisahkan koefisien dari ruas kanan. Fungsi ini dipanggil tiap langkah supaya proses eliminasinya bisa diikuti satu per satu.

In [ ]:
def cetak_papan(papan, judul):                                   # tampilkan matriks gabungan [A | b]
    print(f"\n{judul}")                                          # judul langkah di baris sendiri
    for baris in papan:                                          # satu persamaan per putaran
        kiri = "  ".join(f"{round(v, 10) + 0.0:9.4f}" for v in baris[:-1])   # koefisien; round + 0.0 membuang tampilan -0.0000
        print(f"  [ {kiri} | {round(baris[-1], 10) + 0.0:9.4f} ]")           # ruas kanan setelah garis tegak


### Cell 4 · Alat bantu operasi baris

**Penjelasan:** Lima alat kecil untuk operasi baris: merakit papan, menukar dua baris, mencari pivot terkuat, menjadikan pivot bernilai 1, dan mengosongkan kolom. Kupecah jadi fungsi-fungsi kecil supaya mesin utama di Cell 5 lebih mudah dibaca.

In [ ]:
def rakit_papan(koef, kanan):                                    # gabungkan koefisien dan ruas kanan
    return [list(map(float, baris)) + [float(k)] for baris, k in zip(koef, kanan)]   # ubah ke desimal

def tukar_dua_baris(papan, i, j):                                # tukar posisi baris i dan j
    papan[i], papan[j] = papan[j], papan[i]                      # tukar langsung tanpa variabel bantu

def cari_baris_terkuat(papan, kolom, mulai):                     # cari pivot terbesar di satu kolom
    terbaik = mulai                                              # anggap baris awal yang terbesar dulu
    for r in range(mulai + 1, len(papan)):                       # periksa baris-baris di bawahnya
        if abs(papan[r][kolom]) > abs(papan[terbaik][kolom]):    # bandingkan nilai mutlak
            terbaik = r                                          # simpan baris yang lebih besar
    return terbaik                                               # kembalikan nomor barisnya

def jadikan_satu(papan, r, kolom):                               # bagi satu baris supaya pivot = 1
    pembagi = papan[r][kolom]                                    # nilai pivot saat ini
    papan[r] = [v / pembagi for v in papan[r]]                   # bagi semua isi baris dengan pivot

def kosongkan_kolom(papan, r_acuan, kolom):                      # nolkan kolom di semua baris lain
    for r in range(len(papan)):                                  # periksa tiap baris
        if r == r_acuan:                                         # baris acuan jangan diubah
            continue                                             # lompat ke baris berikutnya
        faktor = papan[r][kolom]                                 # angka yang harus dihilangkan
        papan[r] = [v - faktor * a for v, a in zip(papan[r], papan[r_acuan])]   # kurangi kelipatan baris acuan


### Cell 5 · Mesin Gauss-Jordan

**Penjelasan:** Ini inti programnya. Untuk setiap kolom: cari pivot terkuat, tukar baris kalau perlu, jadikan pivot bernilai 1, lalu nolkan kolom itu di baris lain. Kalau pivotnya hampir nol, berarti SPL tidak punya solusi tunggal, jadi program berhenti dengan pesan yang jelas. Parameter `tampil_langkah` mengatur apakah papan tiap langkah dicetak.

In [ ]:
BATAS_NOL = 1e-12                                                # di bawah angka ini dianggap nol

def selesaikan_spl(koef, kanan, tampil_langkah=True):            # mesin utama Gauss-Jordan
    papan = rakit_papan(koef, kanan)                             # bentuk matriks gabungan
    n = len(papan)                                               # jumlah persamaan (3)
    if tampil_langkah:                                           # kalau langkah ingin ditampilkan
        cetak_papan(papan, "Papan awal")                         # tunjukkan kondisi awal
    for kol in range(n):                                         # kerjakan kolom satu per satu
        r_kuat = cari_baris_terkuat(papan, kol, kol)             # cari calon pivot terbesar
        if abs(papan[r_kuat][kol]) < BATAS_NOL:                  # pivot hampir nol?
            raise ValueError(f"Kolom ke-{kol + 1} tidak punya pivot: SPL tidak punya solusi tunggal.")   # hentikan
        if r_kuat != kol:                                        # pivot belum di posisi yang benar?
            tukar_dua_baris(papan, kol, r_kuat)                  # tukar baris
            if tampil_langkah:                                   # tampilkan hasil tukar
                cetak_papan(papan, f"Tukar baris {kol + 1} dengan baris {r_kuat + 1}")
        jadikan_satu(papan, kol, kol)                            # pivot dijadikan 1
        if tampil_langkah:                                       # tampilkan hasil normalisasi
            cetak_papan(papan, f"Baris {kol + 1} dibagi pivot (pivot jadi 1)")
        kosongkan_kolom(papan, kol, kol)                         # nolkan kolom di baris lain
        if tampil_langkah:                                       # tampilkan hasil pengosongan
            cetak_papan(papan, f"Kolom {kol + 1} dikosongkan di baris lain")
    return [papan[r][-1] for r in range(n)]                      # kolom terakhir = jawaban


### Cell 6 · Jalankan dan lihat hasilnya

**Penjelasan:** Sekarang mesinnya kujalankan pada SPL dari Cell 2. Layar akan menampilkan papan di setiap langkah, dari papan awal sampai bentuk akhir `[I | hasil]`. Di bagian bawah, nilai p, q, r dicetak.

In [ ]:
hasil = selesaikan_spl(koefisien_a, ruas_kanan_b)               # jalankan dengan tampilan langkah
print("\nJawaban akhir:")                                         # pemisah antar bagian
for nama, nilai in zip(nama_peubah, hasil):                      # pasangkan nama variabel dan nilainya
    print(f"  {nama} = {nilai:.4f}")                             # cetak dengan 4 angka desimal



Papan awal
  [    3.0000     5.0000    -2.0000 |  -17.0000 ]
  [    6.0000    -1.0000     4.0000 |   55.0000 ]
  [   -2.0000     7.0000     3.0000 |   -8.0000 ]

Tukar baris 1 dengan baris 2
  [    6.0000    -1.0000     4.0000 |   55.0000 ]
  [    3.0000     5.0000    -2.0000 |  -17.0000 ]
  [   -2.0000     7.0000     3.0000 |   -8.0000 ]

Baris 1 dibagi pivot (pivot jadi 1)
  [    1.0000    -0.1667     0.6667 |    9.1667 ]
  [    3.0000     5.0000    -2.0000 |  -17.0000 ]
  [   -2.0000     7.0000     3.0000 |   -8.0000 ]

Kolom 1 dikosongkan di baris lain
  [    1.0000    -0.1667     0.6667 |    9.1667 ]
  [    0.0000     5.5000    -4.0000 |  -44.5000 ]
  [    0.0000     6.6667     4.3333 |   10.3333 ]

Tukar baris 2 dengan baris 3
  [    1.0000    -0.1667     0.6667 |    9.1667 ]
  [    0.0000     6.6667     4.3333 |   10.3333 ]
  [    0.0000     5.5000    -4.0000 |  -44.5000 ]

Baris 2 dibagi pivot (pivot jadi 1)
  [    1.0000    -0.1667     0.6667 |    9.1667 ]
  [    0.0000     1

### Cell 7 · Cek jawaban dengan substitusi

**Penjelasan:** Supaya yakin jawabannya benar, hasil dimasukkan lagi ke tiap persamaan. Kalau selisih ruas kiri dan ruas kanan nyaris nol, berarti jawabannya cocok. Sisa selisih kecil muncul karena komputer menghitung dengan bilangan desimal.

In [ ]:
sisa = []                                                       # tempat menampung selisih tiap persamaan
for baris, kanan in zip(koefisien_a, ruas_kanan_b):              # ambil persamaan satu per satu
    kiri = sum(k * v for k, v in zip(baris, hasil))              # hitung ruas kiri memakai jawaban
    sisa.append(kiri - kanan)                                    # simpan selisihnya
    print(f"ruas kiri = {kiri:9.5f} | ruas kanan = {kanan:4d} | selisih = {kiri - kanan:.2e}")   # tampilkan
print("Semua persamaan terpenuhi?", all(abs(s) < 1e-9 for s in sisa))   # True jika semua selisih sangat kecil


ruas kiri = -17.00000 | ruas kanan =  -17 | selisih = -7.11e-15
ruas kiri =  55.00000 | ruas kanan =   55 | selisih = 0.00e+00
ruas kiri =  -8.00000 | ruas kanan =   -8 | selisih = 5.33e-15
Semua persamaan terpenuhi? True


### Cell 8 · Uji kasus tanpa solusi tunggal

**Penjelasan:** Program yang baik harus tahu kapan menyerah. Di sini kuberi SPL yang baris kedua adalah kelipatan dua dari baris pertama, jadi persamaannya "kembar" dan tidak ada solusi tunggal. Program harus berhenti dengan pesan galat, bukan menghasilkan angka ngawur.

In [ ]:
koef_kembar = [[1, 2, 3], [2, 4, 6], [1, 0, 1]]                 # baris 2 = 2 x baris 1
kanan_kembar = [4, 8, 2]                                         # ruas kanan untuk tiga persamaan tadi
try:                                                             # coba jalankan mesin utama
    selesaikan_spl(koef_kembar, kanan_kembar, tampil_langkah=False)   # tanpa tampilan langkah
except ValueError as galat:                                      # tangkap pesan galat dari mesin
    print("Program berhenti dengan pesan:", galat)               # tampilkan pesannya


Program berhenti dengan pesan: Kolom ke-3 tidak punya pivot: SPL tidak punya solusi tunggal.


### Cell 9 · Uji acak: bandingkan dengan NumPy

**Penjelasan:** Uji terakhir: aku membuat tiga SPL acak. Untuk tiap SPL, kutentukan dulu jawaban "kunci" yang acak, lalu kuhitung ruas kanannya. Mesinku harus menebak kunci itu kembali, dan hasilnya kubandingkan juga dengan `numpy.linalg.solve`. `random.seed` dikunci supaya hasilnya sama setiap dijalankan.

In [ ]:
random.seed(20251110)                                           # kunci keacakan agar hasil bisa diulang

def buat_spl_acak():                                             # buat matriks 3x3 acak yang punya solusi tunggal
    while True:                                                  # ulang sampai dapat yang cocok
        a = [[random.randint(-9, 9) for _ in range(3)] for _ in range(3)]   # angka acak -9 sampai 9
        if abs(np.linalg.det(np.array(a))) > 1e-6:               # determinan tidak nol = ada solusi tunggal
            return a                                             # kembalikan matriksnya

for uji_ke in range(1, 4):                                       # tiga kali percobaan
    a = buat_spl_acak()                                          # matriks koefisien acak
    kunci = [random.randint(-9, 9) for _ in range(3)]            # jawaban rahasia yang harus ditebak
    b = [sum(k * v for k, v in zip(baris, kunci)) for baris in a]   # ruas kanan sesuai jawaban rahasia
    tebakan = selesaikan_spl(a, b, tampil_langkah=False)         # tebakan dari mesin Gauss-Jordan
    cocok = np.allclose(tebakan, np.linalg.solve(np.array(a, float), np.array(b, float)))   # bandingkan dengan NumPy
    print(f"Uji {uji_ke}: kunci = {kunci} | tebakan = {[round(t, 4) + 0.0 for t in tebakan]} | cocok dengan NumPy: {cocok}")


Uji 1: kunci = [9, 8, 0] | tebakan = [9.0, 8.0, 0.0] | cocok dengan NumPy: True
Uji 2: kunci = [-8, -9, -5] | tebakan = [-8.0, -9.0, -5.0] | cocok dengan NumPy: True
Uji 3: kunci = [9, 7, 6] | tebakan = [9.0, 7.0, 6.0] | cocok dengan NumPy: True
